# Notebook 05 — Deep Learning Portfolio Construction (Exploration)

*Brandt-Santa-Clara-Valkanov (2009) parametric portfolio policies on a 2-asset SPY+IEF universe with daily and intraday-derived features.*

## Abstract

This exploration applies the Brandt, Santa-Clara & Valkanov (2009) parametric portfolio policy framework to a 2-asset universe (SPY + IEF, equity + 10-year Treasury) using deep learning policies (MLP, LSTM, Transformer) trained end-to-end on portfolio-level objectives. The central question: does adding intraday-derived realized power as a feature improve out-of-sample performance over daily features alone? We test three loss variants (Sharpe, CRRA utility with γ=5, CRRA with shrinkage to a risk-parity benchmark), three architectures, and two feature variants — 18 configurations averaged across 10 seeds each — on the 2021-2026 sample period covered by EODHD intraday data. The notebook is a standalone exploration; the main comparative paper's empirical conclusions (Sessions 1-3) are not affected by these results.

## §T1 Theoretical Framework — BSV (2009)

### Direct Portfolio Policy Parameterization

Brandt, Santa-Clara & Valkanov (2009) propose mapping asset characteristics directly to portfolio weights without an intermediate forecasting step:

$$w_{i,t} = f_\theta(x_{i,t})$$

where $x_{i,t}$ is the vector of observable characteristics for asset $i$ at time $t$, and $\theta$ are the policy parameters. The portfolio objective is maximized end-to-end:

$$\max_\theta \frac{1}{T}\sum_{t=1}^{T} U(R_{p,t+1}), \qquad R_{p,t+1} = \sum_{i=1}^{N} w_{i,t}(\theta)\, r_{i,t+1}$$

This directly avoids the noise amplification of mean-variance optimization on small-sample returns — a classic two-step approach that first estimates $\hat{\mu}$ and then plugs it into a quadratic program. Here $\theta$ is trained on the full portfolio return distribution, so the optimizer never touches a separate covariance estimate. A recent industry implementation that informed our architecture and feature choices is Cheng & Wu (2024).

**Reference:** Brandt, M. W., Santa-Clara, P., & Valkanov, R. (2009). Parametric portfolio policies: Exploiting characteristics in the cross-section of equity returns. *Review of Financial Studies*, 22(9), 3411–3447.

## §T2 Loss Functions

We compare three portfolio-level training objectives:

### Sharpe Loss

$$\mathcal{L}_{\text{Sharpe}}(\theta) = -\frac{\hat{\mu}(R_p)}{\hat{\sigma}(R_p)}$$

The empirical Sharpe ratio of the training-batch portfolio returns, negated for minimization.

### CRRA Utility Loss (γ = 5)

$$\mathcal{L}_{\text{CRRA}}(\theta) = -\frac{1}{T}\sum_{t=1}^{T} \frac{(1 + R_{p,t})^{1-\gamma}}{1-\gamma}$$

The Taylor expansion of CRRA produces alternating signs on odd/even moments, implicitly rewarding higher returns and positive skewness while penalizing variance and kurtosis. γ=5 corresponds to moderately high risk aversion.

### CRRA + Shrinkage to Risk Parity

$$w_{i,t} = \sigma(f_\theta(x_{i,t})) \cdot w_i^{\text{RP}}$$

where $\sigma(\cdot)$ is the sigmoid function and $w^{\text{RP}}$ is the 21-day vol-weighted risk-parity benchmark weight. The sigmoid output acts as a per-asset multiplier in $(0,1)$, shrinking freely toward the benchmark. The CRRA utility with γ=5 is then applied to the effective weights.

**Reference:** Brandt, M. W. (1999). Estimating portfolio and consumption choice: A conditional Euler equations approach. *Journal of Finance*, 54(5), 1609–1645.

## §T3 Feature Design — Realized Power

### Definition

$$\text{RP}_t = \sum_k r_{t,k}^2$$

where $r_{t,k}$ are intraday 5-minute log returns within trading day $t$, restricted to regular hours (09:30–16:00 ET). Each trading day contributes approximately 78 5-minute bars, yielding ~78 squared returns summed into a single scalar.

### Why Realized Power?

Realized power is a noise-robust daily volatility proxy directly derived from high-frequency data. Unlike daily volatility (which is the squared daily return — a single noisy observation per day), realized power aggregates many sub-daily observations, dramatically reducing measurement error. The question this notebook asks is whether replacing (or augmenting) daily volatility estimates with realized power as an input feature changes the policy's out-of-sample performance on the 2024–2026 test window.

A recent industry implementation using intraday-derived features in direct-weight DL portfolio policies is Cheng & Wu (2024): *Big Data and AI Strategies: Deep Learning Portfolio Construction*, J.P. Morgan Global Quantitative & Derivatives Strategy, 14 March 2024. That work informed our architecture choices (attention-based encoders, realized power features, 10-seed ensemble averaging); BSV (2009) is the primary methodological framework.

## §T4 What This Exploration Adds

- **Three architecture families** (MLP / LSTM / Transformer) evaluated in parallel under both feature variants and all three loss functions = **18 configs × 10 seeds = 180 policies**.
- **Direct feature ablation**: daily-only vs daily+realized-power, holding architecture, loss, hyperparameters, and seeds identical.
- **Three benchmarks**: risk parity (21-day vol-weighted, daily rebalance), 60/40 static, equal-weight 50/50.
- **Self-contained on our 5.3-year SPY+IEF window** (2021-01-04 → 2026-04-30) using our cached intraday data.
- Reproducibility: cached parquet at `data/cache/intraday_5min_SPY_IEF_2021_2026.parquet`; published CSV at `data/published/intraday_5min_SPY_IEF_2021_2026.csv`.

---
## §0 Setup

In [ ]:
import os, sys, warnings
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.environ.setdefault("OMP_NUM_THREADS", "1")
warnings.filterwarnings("ignore", category=FutureWarning)

# ── device ──────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

# ── repo root on sys.path ────────────────────────────────────────────────────
ROOT = Path("__file__").parent.parent if "__file__" in dir() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# ── experiment constants ─────────────────────────────────────────────────────
ASSETS      = ["SPY.US", "IEF.US"]
TRAIN_END   = "2023-12-31"
VAL_START   = "2024-01-01"
VAL_END     = "2024-06-30"
TEST_START  = "2024-07-01"
TEST_END    = "2026-04-30"
SEEDS       = tuple(range(10))
TRADING_DAYS = 252

# Smoke-mode overrides (uncomment for a quick end-to-end check):
# SEEDS = (0,)
# MAX_EPOCHS = 5; PATIENCE = 2
MAX_EPOCHS = 80
PATIENCE   = 12

OUT_DIR = ROOT / "results" / "notebook_05"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output dir: {OUT_DIR}")
print(f"Seeds: {SEEDS}  |  max_epochs: {MAX_EPOCHS}  |  patience: {PATIENCE}")

## §1 Data Loading

In [ ]:
from aiam.data.intraday import load_cached_intraday

# ── daily prices ─────────────────────────────────────────────────────────────
prices_all = pd.read_parquet(ROOT / "data/cache/prices_29.parquet")
prices = prices_all[ASSETS].copy()
prices.index = pd.to_datetime(prices.index)

# Restrict to intraday-covered period
prices = prices.loc["2021-01-04":"2026-04-30"]
print(f"Daily prices: {prices.shape}  |  {prices.index.min().date()} → {prices.index.max().date()}")

# ── intraday panel ────────────────────────────────────────────────────────────
from datetime import date
intraday = load_cached_intraday(
    ROOT / "data/cache/intraday_5min_SPY_IEF_2021_2026.parquet",
    tickers=ASSETS,
)
print(f"Intraday bars: {intraday.shape}  |  assets: {intraday.index.get_level_values('asset').unique().tolist()}")

## §2 Daily Feature Engineering

In [ ]:
# ── daily log returns ─────────────────────────────────────────────────────────
daily_ret = np.log(prices / prices.shift(1)).dropna()

def _rolling(ret, window, fn):
    return ret.rolling(window).apply(fn, raw=True)

mom_5   = daily_ret.rolling(5).sum().shift(1)
mom_21  = daily_ret.rolling(21).sum().shift(1)
mom_63  = daily_ret.rolling(63).sum().shift(1)
vol_60  = daily_ret.rolling(60).std().shift(1)
vol_252 = daily_ret.rolling(252).std().shift(1)

# ── stack into long-format MultiIndex (date, asset) ──────────────────────────
feat_dict = {
    "ret":    daily_ret,
    "mom_5":  mom_5,
    "mom_21": mom_21,
    "mom_63": mom_63,
    "vol_60": vol_60,
    "vol_252":vol_252,
}
feat_long = pd.concat({k: v.stack() for k, v in feat_dict.items()}, axis=1)
feat_long.index.names = ["date", "asset"]
feat_long = feat_long.dropna(subset=["ret", "mom_63", "vol_252"])

DAILY_FEATURES = ["ret", "mom_5", "mom_21", "mom_63", "vol_60", "vol_252"]
print(f"Daily feature panel: {feat_long.shape}  |  features: {DAILY_FEATURES}")
print(f"Date range: {feat_long.index.get_level_values('date').min().date()} → {feat_long.index.get_level_values('date').max().date()}")

## §3 Realized Power Feature

In [ ]:
from aiam.features.realized_power import compute_realized_power_5min

rp_wide = compute_realized_power_5min(intraday, tickers=ASSETS)
rp_wide.index = pd.to_datetime(rp_wide.index)
print(f"Realized power wide: {rp_wide.shape}")
print(f"Date range: {rp_wide.index.min().date()} → {rp_wide.index.max().date()}")
print("\nSample (first 3 rows):")
print(rp_wide.head(3))

# Stack to long format matching feat_long
rp_long = rp_wide.shift(1).stack()  # 1-day lag (no look-ahead)
rp_long.index.names = ["date", "asset"]
rp_long.name = "rp_daily"

print(f"\nRealized power long (lagged): {rp_long.shape}")

## §4 Build the Two Feature Panels

In [ ]:
# ── Panel A: daily-only (6 features) ─────────────────────────────────────────
panel_daily = feat_long[DAILY_FEATURES].copy()

# ── Panel B: daily + realized power (7 features) ──────────────────────────────
panel_rp = feat_long[DAILY_FEATURES].join(rp_long, how="inner")
panel_rp = panel_rp.dropna()

RP_FEATURES = DAILY_FEATURES + ["rp_daily"]

# ── Align both panels to the same date range ──────────────────────────────────
common_dates = panel_daily.index.get_level_values("date").intersection(
    panel_rp.index.get_level_values("date")
)
d_idx_daily = panel_daily.index.get_level_values("date")
d_idx_rp    = panel_rp.index.get_level_values("date")
panel_daily = panel_daily.loc[d_idx_daily.isin(common_dates)]
panel_rp    = panel_rp.loc[d_idx_rp.isin(common_dates)]

print(f"Panel A (daily-only):       {panel_daily.shape}  features={DAILY_FEATURES}")
print(f"Panel B (daily+RP):         {panel_rp.shape}  features={RP_FEATURES}")
print(f"Common date range: {common_dates.min().date()} → {common_dates.max().date()}")

## §5 Target Construction and Splits

In [ ]:
# 1-day forward returns (daily rebalance per BSV / Cheng-Wu framework)
fwd_ret_wide = daily_ret.shift(-1)  # ret[t] = r_{t+1} known at end of day t
fwd_ret_long = fwd_ret_wide.stack()
fwd_ret_long.index.names = ["date", "asset"]
fwd_ret_long.name = "fwd_ret_1d"

# Align target to common_dates
fwd_date_idx = fwd_ret_long.index.get_level_values("date")
target_series = fwd_ret_long.loc[fwd_date_idx.isin(common_dates)].dropna()

# ── Chronological splits ─────────────────────────────────────────────────────
# Train: 2021-01 → 2023-12, Val: 2024-01 → 2024-06, Test: 2024-07 → 2026-04
all_dates = common_dates.sort_values().unique()
train_dates = all_dates[all_dates <= pd.Timestamp(TRAIN_END)]
val_dates   = all_dates[(all_dates >= pd.Timestamp(VAL_START)) & (all_dates <= pd.Timestamp(VAL_END))]
test_dates  = all_dates[all_dates >= pd.Timestamp(TEST_START)]

print(f"Train dates: {len(train_dates)}  ({train_dates.min().date()} → {train_dates.max().date()})")
print(f"Val dates:   {len(val_dates)}  ({val_dates.min().date()} → {val_dates.max().date()})")
print(f"Test dates:  {len(test_dates)}  ({test_dates.min().date()} → {test_dates.max().date()})")

## §6 Benchmark Strategies

In [ ]:
# ── Test-period daily returns for SPY and IEF ─────────────────────────────────
test_daily_ret = daily_ret.loc[test_dates]

# ── Equal-weight 50/50 ────────────────────────────────────────────────────────
ret_ew = test_daily_ret.mean(axis=1)
ret_ew.name = "EW-50/50"

# ── 60/40 (SPY 60%, IEF 40%) ─────────────────────────────────────────────────
ret_6040 = test_daily_ret["SPY.US"] * 0.60 + test_daily_ret["IEF.US"] * 0.40
ret_6040.name = "60/40"

# ── Risk Parity (21-day vol-weighted, daily rebalance) ────────────────────────
# Using full sample vol to compute weights day by day (no look-ahead in vol estimate)
roll_vol = daily_ret.rolling(21).std().shift(1)  # lagged, available at start of day
inv_vol = 1.0 / roll_vol.clip(lower=1e-8)
rp_weights = inv_vol.div(inv_vol.sum(axis=1), axis=0)
ret_rp = (test_daily_ret * rp_weights.loc[test_dates]).sum(axis=1)
ret_rp.name = "RiskParity-21d"

BENCHMARKS = {"RiskParity-21d": ret_rp, "60/40": ret_6040, "EW-50/50": ret_ew}

def ann_sharpe(r):
    r = r.dropna()
    return r.mean() / r.std() * np.sqrt(TRADING_DAYS)

def ann_return(r):
    r = r.dropna()
    return (1 + r).prod() ** (TRADING_DAYS / len(r)) - 1

def ann_vol(r):
    return r.dropna().std() * np.sqrt(TRADING_DAYS)

def max_drawdown(r):
    cum = (1 + r.dropna()).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

print("Benchmark stats (test period):")
for name, r in BENCHMARKS.items():
    print(f"  {name:20s}  Sharpe={ann_sharpe(r):.3f}  Ann Ret={ann_return(r):.1%}  MaxDD={max_drawdown(r):.1%}")

## §7 Training — 18 Configs × 10 Seeds = 180 Policies

In [ ]:
import itertools
import time
from aiam.dl.losses import sharpe_loss, crra_loss, crra_shrinkage_loss
from aiam.dl.policies import MLPPolicy, LSTMPolicy, TransformerPolicy
from aiam.dl.policy_workflow import (
    fit_direct_weight_seed_ensemble,
    build_policy_sequence_windows,
    DirectWeightSeedEnsemble,
)
from aiam.ml.workflow import fit_standardizer, apply_standardizer

ARCHITECTURES = ["MLP", "LSTM", "Transformer"]
FEATURE_VARIANTS = ["daily", "daily+rp"]
LOSS_VARIANTS = ["sharpe", "crra", "crra_shrinkage"]
LOOKBACK = 21  # ~1 trading month; cheaper on small dataset than 63

# ── Risk parity benchmark weights for crra_shrinkage ──────────────────────────
rp_w_train = rp_weights.loc[train_dates].mean()
benchmark_w_tensor = torch.tensor(rp_w_train[ASSETS].values, dtype=torch.float32)

def make_loss_fn(loss_kind):
    if loss_kind == "sharpe":
        return sharpe_loss
    if loss_kind == "crra":
        return lambda w, r: crra_loss(w, r, gamma=5.0)
    if loss_kind == "crra_shrinkage":
        bw = benchmark_w_tensor.clone()
        return lambda w, r: crra_shrinkage_loss(w, r, bw, gamma=5.0)
    raise ValueError(loss_kind)

def get_activation(loss_kind):
    return "sigmoid" if loss_kind == "crra_shrinkage" else "relu"

def get_panel(feat_variant):
    return (panel_daily, DAILY_FEATURES) if feat_variant == "daily" else (panel_rp, RP_FEATURES)

# ── Build tabular arrays (MLP) from standardized features ─────────────────────
def build_tabular_Xy(panel, feature_cols, split_dates):
    """Return (X, y) for tabular MLP: one row per (date, asset), y = all-asset returns."""
    date_idx = panel.index.get_level_values("date")
    target_date_idx = target_series.index.get_level_values("date")

    # Wide target: (date, n_assets)
    tgt_sub = target_series.loc[target_date_idx.isin(split_dates)]
    wide_y = tgt_sub.unstack(level=1).reindex(columns=ASSETS).dropna()
    valid_dates = set(wide_y.index)

    fp_sub = panel.loc[date_idx.isin(valid_dates), feature_cols]
    fp_reset = fp_sub.reset_index().rename(columns={"date": "Date", "asset": "asset"})
    fp_reset = fp_reset.sort_values(["Date", "asset"]).reset_index(drop=True)

    X_rows, y_rows = [], []
    for _, row in fp_reset.iterrows():
        d = pd.Timestamp(row["Date"])
        if d not in wide_y.index:
            continue
        X_rows.append(row[feature_cols].to_numpy(dtype="float32"))
        y_rows.append(wide_y.loc[d].to_numpy(dtype="float32"))

    if not X_rows:
        return (np.empty((0, len(feature_cols)), "float32"),
                np.empty((0, len(ASSETS)), "float32"))
    return np.stack(X_rows), np.stack(y_rows)


# ── Build sequence windows (LSTM / Transformer) ───────────────────────────────
def build_seq_Xy(panel, feature_cols, split_dates):
    return build_policy_sequence_windows(
        panel, target_series, feature_cols, ASSETS, LOOKBACK,
        allowed_dates=set(split_dates),
    )


# ── Standardise per feature-variant, fit on train only ───────────────────────
def get_std_panel(panel, feature_cols):
    d_idx = panel.index.get_level_values("date")
    train_sub = panel.loc[d_idx.isin(train_dates), feature_cols]
    center, scale = fit_standardizer(train_sub, feature_cols)
    return apply_standardizer(panel, center, scale, feature_cols), center, scale

std_panels = {}
std_stats  = {}
for fv in FEATURE_VARIANTS:
    pnl, fcols = get_panel(fv)
    sp, ctr, scl = get_std_panel(pnl, fcols)
    std_panels[fv] = sp
    std_stats[fv]  = (ctr, scl, fcols)
    print(f"Standardized panel '{fv}': {sp.shape}")


# ── Main training loop ────────────────────────────────────────────────────────
ENSEMBLES = {}   # key: (arch, feat_variant, loss_variant)

configs = list(itertools.product(ARCHITECTURES, FEATURE_VARIANTS, LOSS_VARIANTS))
print(f"\nTraining {len(configs)} configs × {len(SEEDS)} seeds = {len(configs)*len(SEEDS)} policies...\n")

for arch, fv, lv in configs:
    key = (arch, fv, lv)
    sp = std_panels[fv]
    _, _, feature_cols = std_stats[fv]
    n_features = len(feature_cols)
    activation = get_activation(lv)
    loss_fn = make_loss_fn(lv)

    t0 = time.time()

    if arch == "MLP":
        X_tr, y_tr = build_tabular_Xy(sp, feature_cols, train_dates)
        X_va, y_va = build_tabular_Xy(sp, feature_cols, val_dates)
        ensemble = fit_direct_weight_seed_ensemble(
            MLPPolicy, X_tr, y_tr, X_va, y_va, loss_fn,
            seeds=SEEDS, device=DEVICE,
            n_features=n_features, n_assets=len(ASSETS),
            hidden_dims=(32, 16), dropout=0.10,
            lr=1e-3, weight_decay=1e-4, batch_size=256,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            activation=activation,
        )

    elif arch == "LSTM":
        X_tr, y_tr, _ = build_seq_Xy(sp, feature_cols, train_dates)
        X_va, y_va, _ = build_seq_Xy(sp, feature_cols, val_dates)
        ensemble = fit_direct_weight_seed_ensemble(
            LSTMPolicy, X_tr, y_tr, X_va, y_va, loss_fn,
            seeds=SEEDS, device=DEVICE,
            n_features=n_features, n_assets=len(ASSETS),
            hidden_dim=24, dropout=0.10,
            lr=1e-3, weight_decay=1e-4, batch_size=256,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            activation=activation,
        )

    else:  # Transformer
        X_tr, y_tr, _ = build_seq_Xy(sp, feature_cols, train_dates)
        X_va, y_va, _ = build_seq_Xy(sp, feature_cols, val_dates)
        ensemble = fit_direct_weight_seed_ensemble(
            TransformerPolicy, X_tr, y_tr, X_va, y_va, loss_fn,
            seeds=SEEDS, device=DEVICE,
            n_features=n_features, n_assets=len(ASSETS),
            d_model=32, nhead=4, num_layers=2, dropout=0.10,
            lr=1e-3, weight_decay=1e-4, batch_size=256,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            activation=activation,
        )

    ENSEMBLES[key] = ensemble
    elapsed = time.time() - t0
    print(f"  [{arch:12s} | {fv:10s} | {lv:15s}]  {len(SEEDS)} seeds  {elapsed:.1f}s")

print("\nAll configs trained.")

## §8 Test-Period Weight Inference

In [ ]:
def infer_weights_tabular(ensemble, std_panel, feature_cols, date):
    """For MLP: infer ensemble-averaged weights at a single date."""
    try:
        X_date = std_panel.xs(date, level="date")[feature_cols]
        X_date = X_date.reindex(ASSETS)
        if X_date.isna().any().any():
            return np.array([0.5, 0.5])
    except KeyError:
        return np.array([0.5, 0.5])
    raw = ensemble.predict_weights(X_date.values.astype("float32"))
    mean_raw = raw.mean(axis=0)
    w = np.clip(mean_raw, 0, None)
    return w / max(w.sum(), 1e-12)


def infer_weights_seq(ensemble, std_panel, feature_cols, date):
    """For LSTM/Transformer: build LOOKBACK window per asset, infer weights."""
    windows = []
    for asset in ASSETS:
        try:
            asset_slice = std_panel.xs(asset, level="asset").sort_index()
        except KeyError:
            return np.array([0.5, 0.5])
        pos = asset_slice.index.searchsorted(date)
        if pos >= len(asset_slice) or asset_slice.index[pos] != date:
            return np.array([0.5, 0.5])
        start = pos - LOOKBACK + 1
        if start < 0:
            return np.array([0.5, 0.5])
        window = asset_slice.iloc[start:pos+1][feature_cols].values.astype("float32")
        if window.shape[0] != LOOKBACK:
            return np.array([0.5, 0.5])
        windows.append(window)
    X_batch = np.stack(windows, axis=0)  # (2, LOOKBACK, n_features)
    raw = ensemble.predict_weights(X_batch)
    mean_raw = raw.mean(axis=0)
    w = np.clip(mean_raw, 0, None)
    return w / max(w.sum(), 1e-12)


# ── Build weight time series and returns for all 18 configs ──────────────────
STRATEGY_RETURNS = {}   # name → pd.Series
STRATEGY_WEIGHTS = {}   # name → pd.DataFrame (date × asset)

for (arch, fv, lv), ensemble in ENSEMBLES.items():
    name = f"{arch}|{fv}|{lv}"
    sp = std_panels[fv]
    _, _, feature_cols = std_stats[fv]

    w_rows, ret_rows = [], []
    for d in test_dates:
        if arch == "MLP":
            w = infer_weights_tabular(ensemble, sp, feature_cols, d)
        else:
            w = infer_weights_seq(ensemble, sp, feature_cols, d)
        r = test_daily_ret.loc[d].reindex(ASSETS).values if d in test_daily_ret.index else np.zeros(2)
        port_ret = float(np.dot(w, r))
        w_rows.append({"date": d, ASSETS[0]: w[0], ASSETS[1]: w[1]})
        ret_rows.append({"date": d, "ret": port_ret})

    wdf = pd.DataFrame(w_rows).set_index("date")
    rdf = pd.DataFrame(ret_rows).set_index("date")["ret"]
    rdf.name = name
    STRATEGY_WEIGHTS[name] = wdf
    STRATEGY_RETURNS[name] = rdf

print(f"Built {len(STRATEGY_RETURNS)} strategy return series.")
print(f"Test period: {test_dates.min().date()} → {test_dates.max().date()}  ({len(test_dates)} days)")

## §9 Per-Seed Stability Table

In [ ]:
stability_rows = []

for (arch, fv, lv), ensemble in ENSEMBLES.items():
    sp = std_panels[fv]
    _, _, feature_cols = std_stats[fv]
    per_seed_sharpes = []

    for fit_result in ensemble.fits:
        seed_ensemble = type('SE', (), {
            'fits': [fit_result],
            'predict_weights': lambda self, X, fr=fit_result: fr.model.eval() or __import__('torch').no_grad().__enter__() or fr.model(__import__('torch').tensor(__import__('numpy').asarray(X, dtype='float32'))).detach().numpy()
        })()

        # Use a single-fit DirectWeightSeedEnsemble for this seed
        from aiam.dl.policy_workflow import DirectWeightSeedEnsemble
        single_seed_ens = DirectWeightSeedEnsemble(fits=[fit_result], seeds=(fit_result.summary['seed'],))

        rets = []
        for d in test_dates:
            if arch == "MLP":
                w = infer_weights_tabular(single_seed_ens, sp, feature_cols, d)
            else:
                w = infer_weights_seq(single_seed_ens, sp, feature_cols, d)
            if d in test_daily_ret.index:
                r = test_daily_ret.loc[d].reindex(ASSETS).values
                rets.append(float(np.dot(w, r)))

        s = pd.Series(rets)
        sh = s.mean() / s.std() * np.sqrt(TRADING_DAYS) if len(s) > 1 else np.nan
        per_seed_sharpes.append(sh)

    stability_rows.append({
        "config": f"{arch}|{fv}|{lv}",
        "arch": arch,
        "feat_variant": fv,
        "loss": lv,
        "sharpe_mean": np.nanmean(per_seed_sharpes),
        "sharpe_std": np.nanstd(per_seed_sharpes),
        "sharpe_min": np.nanmin(per_seed_sharpes),
        "sharpe_max": np.nanmax(per_seed_sharpes),
        "per_seed_sharpes": per_seed_sharpes,
    })

stability_df = pd.DataFrame(stability_rows)
print(stability_df[["config", "sharpe_mean", "sharpe_std", "sharpe_min", "sharpe_max"]].to_string(index=False))

## §10 Comparison Table — 21 Strategies

In [ ]:
comp_rows = []

# DL strategies
for name, ret_series in STRATEGY_RETURNS.items():
    arch, fv, lv = name.split("|")
    comp_rows.append({
        "strategy": name,
        "arch": arch,
        "feat_variant": fv,
        "loss": lv,
        "family": f"DL-{fv}",
        "sharpe": ann_sharpe(ret_series),
        "ann_ret": ann_return(ret_series),
        "ann_vol": ann_vol(ret_series),
        "max_dd": max_drawdown(ret_series),
    })

# Benchmarks
for bname, bret in BENCHMARKS.items():
    comp_rows.append({
        "strategy": bname,
        "arch": "Benchmark",
        "feat_variant": "—",
        "loss": "—",
        "family": "Benchmark",
        "sharpe": ann_sharpe(bret),
        "ann_ret": ann_return(bret),
        "ann_vol": ann_vol(bret),
        "max_dd": max_drawdown(bret),
    })

comp_df = pd.DataFrame(comp_rows).sort_values("sharpe", ascending=False).reset_index(drop=True)
comp_df["rank"] = comp_df.index + 1

print(f"Total strategies: {len(comp_df)}")
print(comp_df[["rank", "strategy", "sharpe", "ann_ret", "ann_vol", "max_dd"]].to_string(index=False))

comp_df.to_csv(OUT_DIR / "comparison.csv", index=False)
print(f"\nSaved → {OUT_DIR / 'comparison.csv'}")

## §11 Plots

In [ ]:
# ── Figure 1: Equity Curves — Top-5 DL + 3 Benchmarks ──────────────────────
matplotlib.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

top5_names = comp_df[comp_df["arch"] != "Benchmark"]["strategy"].head(5).tolist()
bench_names = list(BENCHMARKS.keys())

fig, ax = plt.subplots(figsize=(12, 5))
colors_dl   = plt.cm.tab10(np.linspace(0, 0.5, 5))
colors_bm   = ["#333333", "#888888", "#bbbbbb"]

for i, name in enumerate(top5_names):
    cum = (1 + STRATEGY_RETURNS[name].dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=colors_dl[i], lw=1.5, label=name)

for j, bname in enumerate(bench_names):
    bret = BENCHMARKS[bname]
    cum = (1 + bret.dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=colors_bm[j], lw=1.5, ls="--", label=bname)

ax.set_yscale("log")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Wealth (log scale)")
ax.set_title("Equity Curves — Top-5 DL Strategies vs Benchmarks (Test Period)")
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "equity_curves_05.png", dpi=150)
plt.show()
print("Saved equity_curves_05.png")

In [ ]:
# ── Figure 2: Drawdown ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

for i, name in enumerate(top5_names):
    r = STRATEGY_RETURNS[name].dropna()
    cum = (1 + r).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color=colors_dl[i])
    ax.plot(dd.index, dd.values, color=colors_dl[i], lw=1, label=name)

for j, bname in enumerate(bench_names):
    bret = BENCHMARKS[bname].dropna()
    cum = (1 + bret).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, alpha=0.12, color=colors_bm[j])
    ax.plot(dd.index, dd.values, color=colors_bm[j], lw=1, ls="--", label=bname)

ax.set_xlabel("Date")
ax.set_ylabel("Drawdown")
ax.set_title("Underwater Drawdown — Top-5 DL Strategies vs Benchmarks (Test Period)")
ax.legend(fontsize=7, ncol=2, loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "drawdown_05.png", dpi=150)
plt.show()
print("Saved drawdown_05.png")

In [ ]:
# ── Figure 3: Weight Time Series — 6-panel (2 feat × 3 loss), Transformer only
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)

for row_i, fv in enumerate(FEATURE_VARIANTS):
    for col_j, lv in enumerate(LOSS_VARIANTS):
        ax = axes[row_i][col_j]
        name = f"Transformer|{fv}|{lv}"
        if name not in STRATEGY_WEIGHTS:
            ax.set_title(f"{fv} + {lv}\n(missing)", fontsize=8)
            continue
        wdf = STRATEGY_WEIGHTS[name]
        spy_w = wdf[ASSETS[0]]
        ief_w = wdf[ASSETS[1]]
        ax.stackplot(spy_w.index, spy_w.values, ief_w.values,
                     labels=["SPY", "IEF"], colors=["#1f77b4", "#ff7f0e"], alpha=0.8)
        fv_label = "Daily-only" if fv == "daily" else "Daily+RP"
        lv_label = {"sharpe": "Sharpe", "crra": "CRRA", "crra_shrinkage": "CRRA+Shrink"}[lv]
        ax.set_title(f"{fv_label} + {lv_label}", fontsize=9)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Weight", fontsize=8)
        ax.tick_params(axis='x', labelsize=7, rotation=30)

# Shared legend
spy_patch = mpatches.Patch(color="#1f77b4", alpha=0.8, label="SPY")
ief_patch = mpatches.Patch(color="#ff7f0e", alpha=0.8, label="IEF")
fig.legend(handles=[spy_patch, ief_patch], loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Transformer Policy Weight Allocations (Test Period 2024-07 → 2026-04)", fontsize=11)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(FIG_DIR / "weight_timeseries_05.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved weight_timeseries_05.png")

In [ ]:
# ── Figure 4: Seed Sharpe Distribution — Box Plots ───────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))

config_labels = stability_df["config"].tolist()
per_seed_data  = stability_df["per_seed_sharpes"].tolist()
feat_variants_list = stability_df["feat_variant"].tolist()

bp = ax.boxplot(per_seed_data, patch_artist=True, notch=False,
                medianprops={"color": "black", "lw": 1.5})

bm_sharpe_rp = ann_sharpe(BENCHMARKS["RiskParity-21d"])
ax.axhline(bm_sharpe_rp, color="red", ls="--", lw=1.2, label=f"RiskParity Sharpe={bm_sharpe_rp:.2f}")

for patch, fv_label in zip(bp["boxes"], feat_variants_list):
    patch.set_facecolor("#cccccc" if fv_label == "daily" else "#ff9933")
    patch.set_alpha(0.7)

short_labels = [c.replace("daily+rp", "rp").replace("daily", "d").replace("crra_shrinkage", "shrink") for c in config_labels]
ax.set_xticks(range(1, len(config_labels) + 1))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("OOS Sharpe (per seed)")
ax.set_title("Per-Seed OOS Sharpe Distribution — 18 Configs × 10 Seeds")

grey_patch   = mpatches.Patch(color="#cccccc", alpha=0.7, label="Daily-only")
orange_patch = mpatches.Patch(color="#ff9933", alpha=0.7, label="Daily+RP")
ax.legend(handles=[grey_patch, orange_patch, ax.get_lines()[0]], fontsize=8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "seed_sharpe_dist_05.png", dpi=150)
plt.show()
print("Saved seed_sharpe_dist_05.png")

In [ ]:
# ── Figure 5: Feature Ablation — Daily vs Daily+RP, grouped by arch×loss ────
arch_loss_combos = [(a, l) for a in ARCHITECTURES for l in LOSS_VARIANTS]
x = np.arange(len(arch_loss_combos))
width = 0.35

sharpes_daily = []
sharpes_rp    = []
errs_daily    = []
errs_rp       = []

for arch, lv in arch_loss_combos:
    row_d = stability_df[(stability_df["arch"]==arch) & (stability_df["feat_variant"]=="daily") & (stability_df["loss"]==lv)]
    row_r = stability_df[(stability_df["arch"]==arch) & (stability_df["feat_variant"]=="daily+rp") & (stability_df["loss"]==lv)]
    sharpes_daily.append(row_d["sharpe_mean"].values[0] if len(row_d) else np.nan)
    sharpes_rp.append(row_r["sharpe_mean"].values[0] if len(row_r) else np.nan)
    errs_daily.append(row_d["sharpe_std"].values[0] if len(row_d) else 0)
    errs_rp.append(row_r["sharpe_std"].values[0] if len(row_r) else 0)

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, sharpes_daily, width, yerr=errs_daily, label="Daily-only",
               color="#cccccc", edgecolor="black", capsize=4, alpha=0.9)
bars2 = ax.bar(x + width/2, sharpes_rp, width, yerr=errs_rp, label="Daily+RP",
               color="#ff9933", edgecolor="black", capsize=4, alpha=0.9)

xlabels = [f"{a}\n{l[:6]}" for a, l in arch_loss_combos]
ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=8)
ax.set_ylabel("OOS Sharpe (mean ± 1σ across seeds)")
ax.set_title("Effect of Realized Power Feature on OOS Sharpe by Architecture and Loss")
ax.legend(fontsize=9)
ax.axhline(0, color="black", lw=0.8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "feature_ablation_05.png", dpi=150)
plt.show()
print("Saved feature_ablation_05.png")

In [ ]:
# ── Figure 6: Top-12 Strategies Bar Chart ───────────────────────────────────
top12 = comp_df.head(12)

family_colors = {
    "DL-daily+rp": "#ff9933",
    "DL-daily":    "#cccccc",
    "Benchmark":   "#333333",
}

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(
    range(len(top12)), top12["sharpe"],
    color=[family_colors.get(f, "#aaaaaa") for f in top12["family"]],
    edgecolor="black", alpha=0.85
)
for bar, (_, row) in zip(bars, top12.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{row['sharpe']:.2f}\nDD={row['max_dd']:.0%}",
            ha="center", va="bottom", fontsize=6.5)

short_strat_labels = []
for s in top12["strategy"]:
    parts = s.split("|")
    if len(parts) == 3:
        arch, fv, lv = parts
        fv_short = "rp" if fv == "daily+rp" else "d"
        lv_short = {"sharpe":"sh", "crra":"cr", "crra_shrinkage":"crsh"}[lv]
        short_strat_labels.append(f"{arch}\n{fv_short}+{lv_short}")
    else:
        short_strat_labels.append(s)

ax.set_xticks(range(len(top12)))
ax.set_xticklabels(short_strat_labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("OOS Annualized Sharpe")
ax.set_title("Top 12 Strategies — OOS Sharpe with Max Drawdown Annotation")

patches = [mpatches.Patch(color=c, alpha=0.85, label=l) for l, c in family_colors.items()]
ax.legend(handles=patches, fontsize=8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "top_strategies_05.png", dpi=150)
plt.show()
print("Saved top_strategies_05.png")

## §12 Save Artifacts

In [ ]:
# ── Comparison CSV (already saved in §10) ─────────────────────────────────────
print(f"Comparison table: {OUT_DIR / 'comparison.csv'}  ({len(comp_df)} rows)")

# ── Stability CSV ─────────────────────────────────────────────────────────────
stab_save = stability_df.drop(columns=["per_seed_sharpes"]).copy()
stab_save.to_csv(OUT_DIR / "stability.csv", index=False)
print(f"Stability table: {OUT_DIR / 'stability.csv'}  ({len(stab_save)} rows)")

# ── Returns parquet ───────────────────────────────────────────────────────────
ret_df = pd.DataFrame(STRATEGY_RETURNS)
for bname, bret in BENCHMARKS.items():
    ret_df[bname] = bret
ret_df.to_parquet(OUT_DIR / "strategy_returns.parquet")
print(f"Returns parquet: {OUT_DIR / 'strategy_returns.parquet'}  shape={ret_df.shape}")

# ── Weight panels ─────────────────────────────────────────────────────────────
for name, wdf in STRATEGY_WEIGHTS.items():
    safe_name = name.replace("|", "_").replace("+", "")
    wdf.to_parquet(OUT_DIR / f"weights_{safe_name}.parquet")
print(f"Weight parquets saved for {len(STRATEGY_WEIGHTS)} configs.")

print("\nAll artifacts saved.")

## §13 Headline + Findings

In [ ]:
top_dl   = comp_df[comp_df["arch"] != "Benchmark"].iloc[0]
rp_bmark = comp_df[comp_df["strategy"] == "RiskParity-21d"].iloc[0]

# Feature ablation summary
abl = stability_df.copy()
daily_mean = abl[abl["feat_variant"]=="daily"]["sharpe_mean"].mean()
rp_mean    = abl[abl["feat_variant"]=="daily+rp"]["sharpe_mean"].mean()
delta_rp   = rp_mean - daily_mean

# Best config by feature variant
best_daily = abl[abl["feat_variant"]=="daily"].sort_values("sharpe_mean", ascending=False).iloc[0]
best_rp    = abl[abl["feat_variant"]=="daily+rp"].sort_values("sharpe_mean", ascending=False).iloc[0]

print("="*65)
print("  NOTEBOOK 05 HEADLINE — DL Portfolio Construction on SPY+IEF")
print("="*65)
print(f"  Top DL strategy:         {top_dl['strategy']}")
print(f"    OOS Sharpe:            {top_dl['sharpe']:.3f}")
print(f"    Ann. Return:           {top_dl['ann_ret']:.1%}")
print(f"    Ann. Vol:              {top_dl['ann_vol']:.1%}")
print(f"    Max Drawdown:          {top_dl['max_dd']:.1%}")
print()
print(f"  Risk Parity benchmark Sharpe: {rp_bmark['sharpe']:.3f}")
print()
print(f"  Feature ablation (mean across arch×loss configs):")
print(f"    Daily-only Sharpe avg: {daily_mean:.3f}")
print(f"    Daily+RP Sharpe avg:   {rp_mean:.3f}")
print(f"    ΔSharpe (RP - daily):  {delta_rp:+.3f}")
print()
print(f"  Best daily-only config:  {best_daily['config']}  Sharpe={best_daily['sharpe_mean']:.3f}±{best_daily['sharpe_std']:.3f}")
print(f"  Best daily+RP config:    {best_rp['config']}  Sharpe={best_rp['sharpe_mean']:.3f}±{best_rp['sharpe_std']:.3f}")
print()
print(f"  Comparison table rows:   {len(comp_df)} (18 DL + 3 benchmarks)")
print(f"  Total policies trained:  {len(ENSEMBLES) * len(SEEDS)}  ({len(ENSEMBLES)} configs × {len(SEEDS)} seeds)")
print("="*65)
print("  Full results in results/notebook_05/ — populate findings doc")
print("  (docs/handoff/notebook_05_findings.md) with the above numbers.")
print("="*65)